# S3 STARE-PODS Demo — AWS S3 + RDS Postgres

Cloud counterpart of `local_starepods_examples.ipynb`. Runs the same flow against real **AWS S3** (Parquet partitions) and a real **RDS Postgres** `PodsMetadata` table.

**Core workflow**
1. **Ingest granules from four instruments** (GMI, SSMIS, AMSR2, ATMS) → S3 Parquet + RDS metadata (four instruments so the overlap analytics in step 10 can show 2-, 3- and 4-way rendezvous)
2. **Find intersecting data** via STARE SIDs + RDS (bbox filter optional; default loads the full granule)
3. **Download intersecting Parquet partitions** from S3
4. **Reconstitute HDF5** (S1 + S2 scans)
5. **Structure comparison** — reconstituted vs original
6. **RDS metadata verification**

**Temporal features**

7. **Temporal catalog** — every chunk carries `[t_start, t_end]` + podcode
8. **Period-filtered load** — data-level `[t_start, t_end]` overlap
9. **VCF temporal roll-up** — union range per pod, on the fly
10. **Multi-instrument overlap analytics** — 2-, 3- and 4-way rendezvous

**Plots**

11. **Pod coverage map** — the level-4 pods each instrument's chunks occupy
12-14. **One rendezvous up close at each width** (4-, 3- and 2-way) — the swaths (where) beside their pass windows (when)

> The S3/RDS temporal loaders read the **shared** `PodsMetadata` catalog — every ingest in the RDS table, not only this demo's granule (filtered by instrument). That is the production query surface, so the temporal counts reflect the whole catalog, unlike the local demo's fresh isolated SQLite.

**Requires** `starepandas/.config` with AWS + RDS credentials. Sample granules default to the in-repo GMI + SSMIS pair plus the four rendezvous granules; override the pair with `STAREPODS_SAMPLE_GRANULE` / `STAREPODS_SAMPLE_GRANULE_SSMIS`.

In [ ]:
#import subprocess, sys
#subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
#                       "/Users/thatdaihaiton/Workspace/STARE/STAREPandas/stare_demo_and_construct_parallel",
#                       "-q"])

In [ ]:
import os
import time
import h5py
from starepandas.demo_lib import StarePodsDemo
from starepandas.staredataframe import _ensure_rds_db_and_table

### Per-cell timing

Every code cell below reports its own wall time, and `CELL_TIMINGS` keeps them for the summary at the end. Useful for seeing where a run actually spends its time — for these demos it is dominated by ingest, not by any of the queries or analytics.

In [ ]:
import time
from IPython import get_ipython

CELL_TIMINGS = {}
_cell_start = {}


def _time_pre(info):
    _cell_start['t0'] = time.perf_counter()


def _time_post(result):
    t0 = _cell_start.pop('t0', None)
    if t0 is None:                      # cell failed before pre_run_cell ran
        return
    elapsed = time.perf_counter() - t0
    CELL_TIMINGS[result.execution_count] = elapsed
    print(f"\u23f1  cell [{result.execution_count}] took {elapsed:.2f}s")


_ip = get_ipython()
# Re-running this cell would otherwise stack duplicate hooks.
for _event, _fn in (('pre_run_cell', _time_pre), ('post_run_cell', _time_post)):
    _existing = [f for f in _ip.events.callbacks[_event] if f.__name__ == _fn.__name__]
    for _old in _existing:
        _ip.events.unregister(_event, _old)
    _ip.events.register(_event, _fn)

print("per-cell timing enabled")

## Configuration

Edit these paths and parameters before running.

In [ ]:
import pandas as pd
import starepandas

# AWS + RDS credentials. Resolved relative to the installed package so it works
# from any cwd (the .config lives next to the starepandas package).
CONFIG_PATH = os.path.join(os.path.dirname(os.path.abspath(starepandas.__file__)), ".config")

# Resolve the sample granules from the in-repo test-data dir so the notebook is
# safe to run anywhere (no dependency on an external sample directory). Override
# the pair with the STAREPODS_SAMPLE_GRANULE* env vars.
_REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(starepandas.__file__)))
_GRANULE_DIR = os.path.join(_REPO_ROOT, "tests", "data", "granules")

GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE",
    os.path.join(_GRANULE_DIR,
                 "1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5"),
)

# GMI + SSMIS are a co-located pair (both 2025-01-01, concurrent orbits) whose
# ground tracks cross within ~3 min in 42 shared pods — the tightest rendezvous
# in the demo data.
SSMIS_GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE_SSMIS",
    os.path.join(_GRANULE_DIR,
                 "1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5"),
)

# A pair only ever fills the n=2 column of the slide-9 table. These four
# granules — one per instrument, all later the same day — are a verified
# **4-way** rendezvous: over pods q03200 and q03203 the passes arrive
# SSMIS 21:29 -> AMSR2 21:46 -> ATMS 21:47 -> GMI 22:13, i.e. all four within
# ~45 min. Ingesting them alongside the pair populates every cell of the
# slide-8 matrix and the n=2/3/4 columns of the slide-9 table from real data.
RENDEZVOUS_GRANULES = [
    ("SSMIS", os.path.join(_GRANULE_DIR,
              "1C.F18.SSMIS.XCAL2021-V.20250101-S195732-E213923.078446.V07B.HDF5")),
    ("AMSR2", os.path.join(_GRANULE_DIR,
              "1C.GCOMW1.AMSR2.XCAL2016-V.20250101-S201914-E215806.067167.V07A.HDF5")),
    ("ATMS",  os.path.join(_GRANULE_DIR,
              "1C.NOAA21.ATMS.XCAL2023-V.20250101-S201707-E215835.011117.V07A.HDF5")),
    ("GMI",   os.path.join(_GRANULE_DIR,
              "1C.GPM.GMI.XCAL2016-C.20250101-S204910-E222221.061578.V07B.HDF5")),
]

INSTRUMENTS = ["GMI", "SSMIS", "AMSR2", "ATMS"]

# Coincidence window for step 10. The four passes above span ~45 min, so a
# narrower window still shows 2- and 3-way rendezvous but no 4-way.
OVERLAP_DT = pd.Timedelta(minutes=45)

# The RDS catalog is shared with every other ingest, so steps 10-12 scope their
# reads to the chunks this demo actually wrote. The storage root is the only
# filter that can do that: two ingests of the same instrument covering the same
# hours are indistinguishable by Dataset or period. (Scoping by time window
# instead was both leaky — other jobs' granules straddle any window's edges —
# and lossy, since a window that excludes them also clips this demo's own
# passes.) Scoped this way the S3 catalog matches the local one row for row.
DEMO_PATH_PREFIX = S3_PREFIX

# Step 8 splits the two GMI passes at this instant, which lies in the ~7 h gap
# between them (they cover 11:29-13:03 and 20:49-22:22).
PASS_SPLIT = pd.Timestamp("2025-01-01T16:00:00")

# S3 root where Parquet partitions and RDS metadata for this demo live.
S3_PREFIX = "s3://zarrpods/gmi-demo-parquet"

# STARE partition level used for both ingestion and bbox → SIDs lookup.
# Capped at MAX_PARTITION_LEVEL = 4 (~256 cells/granule), the regime
# where each Parquet partition is multi-MB — ideal for S3.
STARE_LEVEL = 4

# Bounding box filter — set to None to reconstitute the full granule
# (matching local_starepods_examples.ipynb), or e.g. (115, -30, 120, -25)
# to restrict to SW Australia / Perth.
BBOX = None   # full granule, no spatial filter — mirrors the local demo

DATASETS = ["GMI_S1", "GMI_S2"]

OUTPUT_HDF5 = "/tmp/gmi_s3_reconstituted.h5"

# Set to True to wipe S3_PREFIX (S3 objects + RDS metadata rows) before
# ingesting. Mirrors the local demo's CLEAN_BEFORE_RUN flag — prevents
# duplicate RDS rows on re-runs. Keep True unless you intentionally
# want to append more granules under the same prefix.
CLEAN_BEFORE_RUN = True

print(f"Granule  : {os.path.basename(GRANULE_FILE)}")
print(f"SSMIS    : {os.path.basename(SSMIS_GRANULE_FILE)}")
for _instrument, _path in RENDEZVOUS_GRANULES:
    print(f"{_instrument:9s}: {os.path.basename(_path)}")
print(f"Datasets : {DATASETS}")
print(f"BBox     : {BBOX}  (None = full granule)")
print(f"S3 root  : {S3_PREFIX}")
print(f"Clean    : {CLEAN_BEFORE_RUN}")

## Step 1 — Ingest GMI, SSMIS, AMSR2 and ATMS granules → S3 Parquet + RDS

In [ ]:
demo = StarePodsDemo(aws_config_path=CONFIG_PATH)

s3_paths = demo.ingest_granules(
    data_path=GRANULE_FILE,
    instrument="GMI",
    s3_prefix=S3_PREFIX,
    level=STARE_LEVEL,
    clean_before_run=CLEAN_BEFORE_RUN,
)
# Every later granule appends — clean_before_run=False so it does NOT wipe
# the data already written to the same prefix.
ssmis_paths = demo.ingest_granules(
    data_path=SSMIS_GRANULE_FILE,
    instrument="SSMIS",
    s3_prefix=S3_PREFIX,
    level=STARE_LEVEL,
    clean_before_run=False,
)
print(f"GMI  : stored {len(s3_paths)} dataset path(s)")
print(f"SSMIS: stored {len(ssmis_paths)} dataset path(s)")

for instrument, path in RENDEZVOUS_GRANULES:
    paths = demo.ingest_granules(
        data_path=path,
        instrument=instrument,
        s3_prefix=S3_PREFIX,
        level=STARE_LEVEL,
        clean_before_run=False,
    )
    print(f"{instrument:5s}: stored {len(paths)} dataset path(s)  ({os.path.basename(path)})")

# Granule basename — used as a substring filter on group_path. Note: as of
# the quaternary pod-code layout (2026-06-14) the S3 layout is FLAT and the
# granule basename is embedded in the chunk *filename*, bracketed by '-':
#   <S3_PREFIX>/<podcode>-<granule_basename>-<dataset>.parquet
# So the old startswith(S3_PREFIX + '/' + basename) scoping no longer matches.
# We use a substring match on the basename instead — which also keeps steps
# 2-5 scoped to this granule now that two GMI granules are ingested.
granule_basename = os.path.splitext(os.path.basename(GRANULE_FILE))[0]
granule_path_marker = f"-{granule_basename}-"   # matches the filename-embedded span

## Step 2 — Find intersecting data via STARE SIDs

In [ ]:
if BBOX is not None:
    location_sids = demo.get_sids_for_bbox(*BBOX, level=STARE_LEVEL)
    print(f"Generated {len(location_sids)} SIDs for bbox {BBOX}")
    intersecting = demo.find_intersecting_data(location_sids, instruments=["GMI"])
    # Scope to our granule. Substring match on the basename, which the flat
    # pod-code layout embeds in the chunk filename (bracketed by '-').
    if not intersecting.empty and "group_path" in intersecting.columns:
        intersecting = intersecting[
            intersecting["group_path"].str.contains(granule_path_marker, regex=False)
        ]
    print(f"Found {len(intersecting)} intersecting metadata row(s).")
else:
    location_sids = None
    intersecting = None
    print("BBOX is None — Step 4 will reconstitute the full granule directly.")

if intersecting is not None and not intersecting.empty:
    intersecting[["Dataset", "grouped_id", "group_path"]].head(8)


## Step 3 — Download intersecting Parquet partitions from S3

In [ ]:
if intersecting is not None and not intersecting.empty:
    data_dict = demo.download_and_analyze(
        intersecting,
        instruments=list(intersecting["Dataset"].unique()),
    )
    for ds_name, sdf in data_dict.items():
        print(f"{ds_name}: {len(sdf)} rows, columns: {list(sdf.columns[:6])} …")
        display(sdf.head(3))
else:
    print("No intersecting partitions to download — Step 4 will read S3 directly.")
    data_dict = {}

## Step 4 — Reconstitute HDF5 (S1 + S2)

In [ ]:
# s3_prefix scope: with CLEAN_BEFORE_RUN=True the bucket only holds this
# granule's data, so passing the broad S3_PREFIX is correct and avoids the
# layout mismatch the old per-granule S3 prefix would create.
recon_path = demo.reconstitute_hdf5(
    dataset=DATASETS,
    output_hdf5_path=OUTPUT_HDF5,
    bbox=BBOX,
    s3_prefix=S3_PREFIX,
)
print(f"Written to: {recon_path}")


## Step 5 — Structure comparison: reconstituted vs original

In [ ]:
def dump_structure(path, label):
    """Print HDF5 group/dataset tree with shapes and dtypes."""
    print(f"\n--- {label} ---")
    with h5py.File(path, "r") as f:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  /{name:50s} {str(obj.shape):20s} {obj.dtype}")
            elif isinstance(obj, h5py.Group) and name != "/":
                print(f"  /{name:50s} Group")
        f.visititems(_visit)

dump_structure(recon_path, f"RECONSTITUTED  ({os.path.basename(recon_path)})")
dump_structure(GRANULE_FILE, f"ORIGINAL       ({os.path.basename(GRANULE_FILE)})")

## Step 6 — RDS metadata verification

In [ ]:
conn = _ensure_rds_db_and_table("StarePodsMetadata")
try:
    with conn.cursor() as cur:
        # Flat pod-code layout: the basename is embedded in the chunk
        # filename, so use a LIKE substring match (not a startswith prefix).
        cur.execute(
            'SELECT "Dataset", COUNT(*) '
            'FROM "PodsMetadata" '
            'WHERE "MetadataJson"->>%s LIKE %s '
            'GROUP BY "Dataset" ORDER BY "Dataset"',
            ("group_path", f"%{granule_path_marker}%"),
        )
        rows = cur.fetchall()
    print(f"RDS scope: group_path contains '{granule_path_marker}'")
    for ds, cnt in rows:
        print(f"  {ds}: {cnt} partition(s)")
finally:
    conn.close()


## Step 7 — Temporal catalog: every chunk carries `[t_start, t_end]` + podcode

`load_s3_temporal_catalog` returns the thin projection the analytics use (`podcode / Dataset / t_start / t_end`) from RDS — never the heavy `MetadataJson`. This reads the **shared** catalog filtered by instrument, so the counts include every GMI/SSMIS ingest in the table, not just this granule.

In [ ]:
from starepandas.io.granules import (
    load_s3_metadata, load_s3_temporal_catalog, load_s3_vcf,
)
from starepandas.overlap import (
    rendezvous_events, overlap_matrix, overlap_pod_table, pair_drilldown,
    pod_drilldown,
)

catalog = pd.concat(
    [load_s3_temporal_catalog(dataset_prefix=instrument) for instrument in INSTRUMENTS],
    ignore_index=True,
)
print(f"Thin catalog ({'+'.join(INSTRUMENTS)}, catalog-wide): {len(catalog)} chunks across "
      f"{catalog['Dataset'].nunique()} datasets")
display(catalog.groupby('Dataset').agg(
    chunks=('podcode', 'size'),
    first_start=('t_start', 'min'),
    last_end=('t_end', 'max'),
))
catalog.head(6)

## Step 8 — Period-filtered load

`load_s3_metadata(..., period=(start, end))` keeps only chunks whose **data-level** range `[t_start, t_end]` overlaps the period (the shared `_period_conditions` index-friendly rewrite — live EXPLAIN confirms a Bitmap Index Scan on `idx_pods_temporal`). Two GMI passes are now ingested, ~9 h apart, so the filter can tell them apart: the same chunks, selected purely on their temporal range rather than on which granule they came from. A window days away returns none.

In [ ]:
gmi = catalog[catalog['Dataset'].str.startswith('GMI')]
passes = {
    "first GMI pass ": gmi[gmi['t_start'] < PASS_SPLIT],
    "second GMI pass": gmi[gmi['t_start'] >= PASS_SPLIT],
}

for label, chunks in passes.items():
    window = (chunks['t_start'].min(), chunks['t_end'].max())
    hit = load_s3_metadata(dataset_prefix='GMI', period=window)
    print(f"{label}: [{window[0]}, {window[1]}]  ({len(chunks)} chunks)")
    print(f"  -> {len(hit)} of {len(gmi)} GMI chunks match this period")

miss_period = (PASS_SPLIT - pd.Timedelta(days=10), PASS_SPLIT - pd.Timedelta(days=9))
miss = load_s3_metadata(dataset_prefix='GMI', period=miss_period)
print(f"9-10 days earlier -> {len(miss)} chunks")

## Step 9 — VCF temporal roll-up

`load_s3_vcf(level, ...)` groups chunks by their level-`level` ancestor pod and returns each pod's union range `[min(t_start), max(t_end)]` + child count — the on-the-fly temporal hierarchy ("Virtual Collection File"), nothing materialized.

In [ ]:
vcf = load_s3_vcf(1, dataset_prefix='GMI')
print(f"{len(vcf)} level-1 VCF nodes for GMI (one per octant subtree)")
vcf

## Step 10 — Multi-instrument overlap analytics

`rendezvous_events` sweeps for passes simultaneously present in a pod within Δt; the matrix / pod-table / drill-downs all aggregate that one events frame. Since the catalog-wide read (step 7) mixes every ingest in the shared RDS table, we scope the sweep to the **two windows this demo wrote** (the `period` pushes into SQL) so the result is this demo's genuine rendezvous.

With four instruments ingested, the slide-9 table gains its **n=3 and n=4** columns. The 4-way is real but tight: over pods `q03200`/`q03203` the passes arrive SSMIS 21:29 → AMSR2 21:46 → ATMS 21:47 → GMI 22:13, spanning ~45 min — so Δt has to be at least that wide before all four count as simultaneous. The first cell below shows that progression.

In [ ]:
demo_catalog = load_s3_temporal_catalog(path_prefix=DEMO_PATH_PREFIX)
print(f"Scoped to {DEMO_PATH_PREFIX}: {len(demo_catalog)} chunks across "
      f"{demo_catalog['Dataset'].nunique()} datasets — this demo's ingests only.")

print("How the coincidence window dt widens what counts as a rendezvous:")
for dt in (pd.Timedelta(minutes=15), pd.Timedelta(minutes=30), OVERLAP_DT):
    ev = rendezvous_events(demo_catalog, dt)
    table = overlap_pod_table(ev)
    by_n = {int(n): int(table[n].gt(0).sum()) for n in table.columns}
    print(f"  dt={str(dt).split()[-1]}  {len(ev):5d} events  "
          f"{ev['podcode'].nunique():4d} pods   pods by n-way: {by_n}")

events = rendezvous_events(demo_catalog, OVERLAP_DT)
pod_table = overlap_pod_table(events)
widest = max(pod_table.columns)

print(f"\nInstrument x instrument matrix — pods where A & B rendezvous (slide 8), dt={OVERLAP_DT}:")
display(overlap_matrix(events))

print('Per-pod n-way combination counts (slide 9), widest rendezvous first:')
display(pod_table.sort_values(sorted(pod_table.columns, reverse=True), ascending=False).head(10))

print(f"{int(pod_table[widest].gt(0).sum())} pods see all {widest} instruments; "
      f"{int(pod_table.get(3, pd.Series(dtype=int)).gt(0).sum())} see a 3-way.")

print('GMI-SSMIS pair drill-down (first 8 shared pods + crossing times):')
display(pair_drilldown(events, 'GMI', 'SSMIS').head(8))

pod = pod_table[pod_table[widest].gt(0)].index[0]
print(f'Pod drill-down for {pod} — every combination meeting there:')
display(pod_drilldown(events, pod).drop(columns='times'))

## Step 11 — Where each instrument flew: pod coverage map

One panel per instrument, shading the level-4 pods its chunks occupy, with the widest-rendezvous pods outlined in black. The orbital geometry the analytics depend on becomes obvious: GMI's 65° inclination confines it to a ±67° band, while the sun-synchronous instruments (SSMIS, AMSR2, ATMS) run pole to pole. That is why AMSR2–ATMS share hundreds of pods — they are near-co-orbiting — while anything involving GMI is scarcer, and why the 4-way lands at a mid-southern latitude.

In [ ]:
from starepandas.demo_plots import (
    plot_pod_coverage, plot_rendezvous, pod_pixels, rendezvous_of_size,
)

quad_pods = list(pod_table[pod_table[widest].gt(0)].index)
fig = plot_pod_coverage(demo_catalog, highlight=quad_pods)

## Step 12 — A 4-way rendezvous up close: where *and* when

A rendezvous needs **both** halves, and one map cannot show them. Left: the swaths inside the pod's trixel — pods are triangles, this one ~500 km across. Right: each instrument's pass window over that pod, all landing inside a single Δt.

Note the granularity: co-location here means *the same level-4 pod*, not the same pixel — the swaths need not physically touch. That is the unit the slide-8/9 analytics count.

Steps 13 and 14 repeat this view for narrower rendezvous. Each picks a pod whose widest rendezvous is **exactly** that many instruments — otherwise a "2-way" would be illustrated with a pod that also holds a 4-way — and among those, the pod whose *least*-covered participant still covers the most pixels, so the picture is legible.

In [ ]:
demo_meta = load_s3_metadata(path_prefix=DEMO_PATH_PREFIX)

pod, meeting, n_way = rendezvous_of_size(events, 4, metadata=demo_meta)
window = (meeting - OVERLAP_DT, meeting + OVERLAP_DT)
passes = pod_pixels(demo, demo_meta, pod, window)

print(f"Pod {pod}: {n_way} instruments, rendezvous completes {meeting}")
for instrument, pixels in sorted(passes.items(), key=lambda kv: kv[1]['timestamp'].min()):
    print(f"  {instrument:6s} {len(pixels):6d} px  "
          f"{pixels['timestamp'].min()} .. {pixels['timestamp'].max()}")

fig = plot_rendezvous(pod, passes, meeting, OVERLAP_DT)

## Step 13 — A 3-way rendezvous up close

The same view for a **trio**. Three instruments crossing one pod inside the window — the n=3 column of the slide-9 table made concrete.

In [ ]:
pod, meeting, n_way = rendezvous_of_size(events, 3, metadata=demo_meta)
window = (meeting - OVERLAP_DT, meeting + OVERLAP_DT)
passes = pod_pixels(demo, demo_meta, pod, window)

print(f"Pod {pod}: {n_way} instruments, rendezvous completes {meeting}")
for instrument, pixels in sorted(passes.items(), key=lambda kv: kv[1]['timestamp'].min()):
    print(f"  {instrument:6s} {len(pixels):6d} px  "
          f"{pixels['timestamp'].min()} .. {pixels['timestamp'].max()}")

fig = plot_rendezvous(pod, passes, meeting, OVERLAP_DT)

## Step 14 — A 2-way rendezvous up close

And a **pair** — the tightest rendezvous in the demo data: the co-located GMI+SSMIS granules, whose ground tracks cross the same pod within about two minutes of each other.

In [ ]:
pod, meeting, n_way = rendezvous_of_size(events, 2, metadata=demo_meta)
window = (meeting - OVERLAP_DT, meeting + OVERLAP_DT)
passes = pod_pixels(demo, demo_meta, pod, window)

print(f"Pod {pod}: {n_way} instruments, rendezvous completes {meeting}")
for instrument, pixels in sorted(passes.items(), key=lambda kv: kv[1]['timestamp'].min()):
    print(f"  {instrument:6s} {len(pixels):6d} px  "
          f"{pixels['timestamp'].min()} .. {pixels['timestamp'].max()}")

fig = plot_rendezvous(pod, passes, meeting, OVERLAP_DT)

## Run time summary

Where the wall clock actually went, slowest cell first.

In [ ]:
timings = pd.Series(CELL_TIMINGS, name='seconds').sort_index()
timings.index.name = 'In [n]'
print(f"{len(timings)} timed cells, {timings.sum():.1f}s total "
      f"({timings.sum() / 60:.1f} min) — excluding this summary cell itself")
display(timings.sort_values(ascending=False).head(10).round(2).to_frame())